In [1]:
## changing directory

import os
(os.getcwd())

import os
os.chdir(r"C:\Users\jland\OneDrive\Emory Essentials\CLASSES Fall 2025\Computing\datasci530fall2025\Lecture 04")

# import libraries
import pandas as pd
import numpy as np


In [16]:
#  loading necessary data


teams = pd.read_csv("data_raw/constructors.csv")
races = pd.read_csv("data_raw/races.csv")
team_results = pd.read_csv("data_raw/constructor_results.csv")   # constructor points per race
results = pd.read_csv("data_raw/results.csv")               # driver-level results per race
drivers = pd.read_csv("data_raw/drivers.csv")
standings = pd.read_csv("data_raw/constructor_standings.csv")

#

In [15]:
# keep only 1981–2020 races
# After reading the codebook, it made the most sense to start with the races file, and then begin using the foreign keys to merge in the other files
races_8120 = races.loc[(races["year"] >= 1981) & (races["year"] <= 2020), ["raceId","year"]]

# attach years to team_results   
results_by_year = (team_results
      .merge(races_8120, on="raceId", how="inner")         # limits to requested years
      .merge(teams[["constructorId","name"]], on="constructorId", how="left"))

# Here's a dataframe with the results by year for each team from 1981 to 2020
results_by_year = results_by_year.sort_values(["points"], ascending=False)
results_by_year

# ensuring name and constructorId are unique
results_by_year["constructorId"].nunique()
results_by_year["name"].nunique()

# Here I'm finding the top 3 teams by total points from 1981 to 2020 in descending order
top_teams = results_by_year.groupby("name")["points"].sum()
top_teams = top_teams.sort_values(ascending=False)
top_teams

# I'm averaging the points of all teams in order to compare them to the top 3 teams
avg_points = top_teams.mean()
avg_points

# Ferrari had the most points with 7,374 points
## Mercedes was second with 5,685 points
### McLaren was third with 5,229.5 points
#### The average points for all teams was approximately 532 points


np.float64(532.2388059701492)

In [ ]:
# Here I'm changing the time frame and only keeping years 2001 to 2020
races_01_20 = races.loc[(races["year"] >= 2001) & (races["year"] <= 2020), ["raceId","year"]]

# Here I'm running the same code as above, but for the years 2001 to 2020 
results_by_year_2 = (team_results
      .merge(races_01_20, on="raceId", how="inner")         # limits to requested years
      .merge(teams[["constructorId","name"]], on="constructorId", how="left"))

# Here's a dataframe of the results by year for each team from 2001 to 2020
results_by_year_2 = results_by_year_2.sort_values(["points"], ascending=False)
results_by_year_2

# making sure both are unique
results_by_year_2["constructorId"].nunique()
results_by_year_2["name"].nunique()

# Here I'm finding the top 3 teams by total points from 2001 to 2020 in descending order
top_teams = results_by_year_2.groupby("name")["points"].sum()
top_teams = top_teams.sort_values(ascending=False)
top_teams

# I'm averaging the points of all teams in order to compare them to the top 3 teams
avg_points_2 = top_teams.mean()
avg_points_2

top_teams
avg_points_2

# Ferrari was the top team from 2001 to 2020 with 5,862 points
## Mercedes was second with 5,685 points
### Red Bull was third with 5,043.5 points
#### The average points across all teams was approximately 786 points. 



np.float64(786.0142857142857)

In [ ]:
## How did rankings change across periods?

## After researching the distinction between points, standings, and position, I decided it's best to normalize the points
## by each year and team. More modern years have more races, so it to compare accurately,  I'm calculating
## points per race for both time periods.


# 1981–2020
res_8120 = results_by_year.loc[results_by_year["year"].between(1981, 2020)]

points_per_race_8120 = (res_8120
    .groupby(["constructorId","name"], as_index=False)
    .agg(points=("points","sum"), races=("year","count"))
    .assign(points_per_race=lambda d: d["points"]/d["races"])
)

# 2001–2020
res_0120 = results_by_year.loc[results_by_year["year"].between(2001, 2020)]

points_per_race_0120 = (res_0120
    .groupby(["constructorId","name"], as_index=False)
    .agg(points=("points","sum"), races=("year","count"))
    .assign(points_per_race=lambda d: d["points"]/d["races"])
)

# Sorting 1981-2020 to see which teams have the highest points per race
sorted_8120 = points_per_race_8120.sort_values("points_per_race", ascending=False)
sorted_8120
#  Print top 5 teams -- Mercedes, Red Bull, Ferrari, Brawn, and Lotus F1 are the top 5 teams from 1981 to 2020 based on points per race
sorted_8120.head(5)

# Sorting 2001-2020 to see which teams have the highest points per race
sorted_0120 = points_per_race_0120.sort_values("points_per_race", ascending=False)
sorted_0120
#  Print top 5 teams -- Mercedes, Red Bull, Ferrari, Brawn, and Lotus F1 are the top 5 teams from 2001 to 2020 based on points per race
sorted_0120.head(5)

### The top 5 are essentially the same for both time periods, except for Ferrari having 5 extra points per year in the 2001-2020 period


,constructorId,name,points,races,points_per_race
24,131,Mercedes,5685.0,215,26.441860
8,9,Red Bull,5043.5,304,16.590461
5,6,Ferrari,5862.0,372,15.758065
22,23,Brawn,172.0,17,10.117647
30,208,Lotus F1,706.0,77,9.168831


In [12]:
# Different Ferrari Drivers

# Here I'm using the loc method to filter only team names that are Ferrari and grabbing the constructorId from that row. 
ferrari_id = teams.loc[teams["name"].str.lower()=="ferrari", "constructorId"].iloc[0]

# Here I'm merging the results and races from 1981 to 2020, and filtering only for Ferrari using the constructorId
# Now I have all races for Ferrari from 1981 to 2020
res_8120 = (results.merge(races_8120, on="raceId", how="inner")
                  .loc[lambda d: d["constructorId"] == ferrari_id])

# These commands show me how many different drivers Ferrari had from 1981 to 2020
n_drivers_ferrari = res_8120["driverId"].nunique()
n_drivers_ferrari

## There were 25 different Ferrari drivers from 1981 to 2020

25

In [ ]:
# Here I'm creating a dataframe that filters only for Ferrari grouped by year and summing the points for each year
# After that I'm sorting the values in descending order to find Ferrari's best year

ferrari_year_pts = (results_by_year.loc[results_by_year["constructorId"] == ferrari_id]
                      .groupby("year", as_index=False)["points"].sum()
                      .sort_values("points", ascending=False))

# I now have a data frame of Ferrari's best years in descending order. The top row is 2018, which was Ferrari's best year with 571 points.
ferrari_year_pts


#Ferrari's best year was 2018. They had 571 points.


,year,points
37,2018,571.0
36,2017,522.0
38,2019,504.0
34,2015,428.0
31,2012,400.0
35,2016,398.0
29,2010,396.0
30,2011,375.0
32,2013,354.0
23,2004,262.0
